In [1]:
from utils import *
import wandb
from datetime import datetime
# from apollo_torch import APOLLOAdamW

In [2]:
config = Config().load(os.path.join("configs", "vit.json"))

In [ ]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [ ]:
import signal

def checkpointModel(stamp, config, model):
    # Block SIGINT for the duration of the save
    original = signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        latestPath = os.path.join("checkpoints", "pretrain", "latest")
        if not os.path.exists(latestPath):
            os.mkdir(latestPath)

        timePath = os.path.join("checkpoints", "pretrain", stamp)
        if not os.path.exists(timePath):
            os.mkdir(timePath)

        for path in [latestPath, timePath]:
            tmp = os.path.join(path, "checkpoint.tmp")
            dst = os.path.join(path, "checkpoint.pt")
            torch.save(model.state_dict(), tmp)
            os.replace(tmp, dst)
            config.save(os.path.join(path, "config.json"))
    finally:
        signal.signal(signal.SIGINT, original)


def trainModel(config, modelClass, datasetClass, resumePath=None, resumeNumber=None, resumeID=None):
    dataset = datasetClass(config.dataset)
    model = modelClass(config.model)

    if resumePath is not None:
        model.load_state_dict(torch.load(resumePath, weights_only=True))

    model = model.to(device)

    print(f"Model has {sum([p.numel() for p in model.parameters()])} parameters")

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learningRate)

    queue = MoCoQueue(dim=config.model.embedDim, size=config.queueSize)
    objective = EmbeddingLoss(temperature=0.04, alpha=0.1)

    train, test = datasetClass.split(dataset, config)
    train = DataLoader(train, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate, prefetch_factor=5, num_workers=2)
    test = DataLoader(test, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate, prefetch_factor=5, num_workers=2)

    testIter = itertoolsBetter(test)

    run = None
    total = 0 if resumeNumber is None else resumeNumber

    stamp = datetime.now().strftime("%Y-%m-%d %H-%M")

    try:
        for epoch in range(config.epochs):
            progress = 0
            for inputsX, inputsY, info, ids in train:
                model.train()
                optimizer.zero_grad()

                _, outputsX = model(inputsX.to(device))
                _, outputsY = model(inputsY.to(device))
                loss = objective(outputsX, outputsY, ids.to(device), queue=queue)

                trainLoss = loss["total"].detach().item()

                loss["total"].backward()
                optimizer.step()

                with torch.no_grad():
                    model.eval()
                    inputsX1, inputsY1, info1, ids1 = next(testIter)

                    _, outputsX1 = model(inputsX1.to(device))
                    _, outputsY1 = model(inputsY1.to(device))
                    loss1 = objective(outputsX1, outputsY1, ids1.to(device), queue=queue)

                    testLoss = loss1["total"].detach().item()

                if run is None:
                    if resumeID is None:
                        run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                    else:
                        run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize(), id=resumeID, resume="allow")
                
                run.log({"Train Loss": trainLoss, 
                         "Train InfoNCE": loss["info"].detach().item(),
                         "Train SigREG": loss["sig"].detach().item(),
                         "Test Loss": testLoss,
                         "Test InfoNCE": loss1["info"].detach().item(),
                         "Test SigREG": loss1["sig"].detach().item(),
                         "Train Perplexity": torch.exp(loss["info"].detach()).item(),
                         "Test Perplexity": torch.exp(loss1["info"].detach()).item(),
                         "Train Recall@1": recallAtK(outputsX, outputsY, k=1, families=ids.to(device), queue=queue),
                         "Test Recall@1": recallAtK(outputsX1, outputsY1, k=1, families=ids1.to(device), queue=queue),
                         "Train Recall@5": recallAtK(outputsX, outputsY, k=5, families=ids.to(device), queue=queue),
                         "Test Recall@5": recallAtK(outputsX1, outputsY1, k=5, families=ids1.to(device), queue=queue),
                         "Train Recall@10": recallAtK(outputsX, outputsY, k=10, families=ids.to(device), queue=queue),
                         "Test Recall@10": recallAtK(outputsX1, outputsY1, k=10, families=ids1.to(device), queue=queue)
                         }, step=total)

                progress += 1
                total += 1

                print(f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}% |  Train Loss: {trainLoss:.2f} | Test Loss: {testLoss:.2f}", end="")

                if total % 1000 == 0:
                    checkpointModel(stamp, config, model)

    except KeyboardInterrupt:
        checkpointModel(stamp, config, model)

In [ ]:
trainModel(config, ViT, PairedImageData)


Loading MyFonts images from dataset ====================
Images converted: 978301/978380

Fonts serialized: 720/3871google/fonts/ofl/kumarone/KumarOne-Regular.ttf execution context too long
Fonts serialized: 939/3871google/fonts/ofl/kumaroneoutline/KumarOneOutline-Regular.ttf execution context too long
Fonts serialized: 2449/3871google/fonts/ofl/notocoloremojicompattest/NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3544/3871

58295 extra bytes in post.stringData array


Fonts serialized: 3871/3871


Fonts serialized: 106/18624

3 extra bytes in post.stringData array


Fonts serialized: 249/18624

...

Fonts serialized: 15039/18624

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.
cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


Fonts serialized: 15687/18624dafont/fonts/wc_wunderbach_rough.otf cannot open resource
Fonts serialized: 16321/18624dafont/fonts/curves.otf division by zero
Fonts serialized: 16913/18624dafont/fonts/toddypaintbrush.otf unknown file format
Fonts serialized: 17738/18624dafont/fonts/clouty.otf 'ascii' codec can't decode byte 0xc3 in position 6: ordinal not in range(128)
Fonts serialized: 18624/18624

Model has 21492980 parameters


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ubuntu/.netrc
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


1 | 965/115294 | 0.837% |  Train Loss: 7.24 | Test Loss: 7.09